# Week 3：USDC 每日转账量（探索 Notebook）

本 Notebook 用于配合 `analyzer.py` 与 `week3/week3_analysis.py` 学习：

- 从 Week2 导出的 `data/usdc_transfers_*.csv` 读取样本
- 计算每日转账总量与每日日志条数
- 用 Plotly 做可视化（与脚本导出的 HTML 口径一致）

> 提示：链上分析里，“日志条数”不等于“人工交易笔数”，同一笔交易可能产生多条 `Transfer` 日志。

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

# 将项目根目录加入 sys.path，便于导入 analyzer
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "analyzer.py").exists():
    PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

from analyzer import (
    ensure_datetime,
    compute_daily_volume,
    compute_daily_tx_count,
    load_usdc_csv,
)

print("项目根目录:", PROJECT_ROOT)

项目根目录: d:\code\claude_code\Blockchain_learning


In [2]:
# 自动选择 data 目录下最新的 usdc_transfers_*.csv
data_dir = PROJECT_ROOT / "data"
candidates = sorted(data_dir.glob("usdc_transfers_*.csv"), key=lambda p: p.stat().st_mtime)
if not candidates:
    raise FileNotFoundError("未找到 usdc_transfers_*.csv，请先运行 week2 练习2 导出数据")

csv_path = candidates[-1]
print("使用 CSV:", csv_path)

df_raw = load_usdc_csv(csv_path)
df = ensure_datetime(df_raw)
df.head()

使用 CSV: d:\code\claude_code\Blockchain_learning\data\usdc_transfers_1775815222.csv


,transaction_hash,block_number,log_index,from,to,value_raw,value_usdc,timestamp,datetime,date
0,fd3475fbf520702eaf41ec2d1e52951b1842a6c432cf11...,24848486,14,0x000000000004444c5dc75cB358380D2e3dE08A90,0xC165C9A4Db89aa72b23f66b96731A960084a1FE2,91805254,91.805254,1775813999,2026-04-10 09:39:59+00:00,2026-04-10
1,fd3475fbf520702eaf41ec2d1e52951b1842a6c432cf11...,24848486,16,0xC165C9A4Db89aa72b23f66b96731A960084a1FE2,0xE0554a476A092703abdB3Ef35c80e0D76d32939F,91805254,91.805254,1775813999,2026-04-10 09:39:59+00:00,2026-04-10
2,6a16ed04514d75baec0455a0901ee57beefcad80166ae3...,24848486,37,0x000000000004444c5dc75cB358380D2e3dE08A90,0xC165C9A4Db89aa72b23f66b96731A960084a1FE2,90720683,90.720683,1775813999,2026-04-10 09:39:59+00:00,2026-04-10
3,6a16ed04514d75baec0455a0901ee57beefcad80166ae3...,24848486,39,0xC165C9A4Db89aa72b23f66b96731A960084a1FE2,0xE0554a476A092703abdB3Ef35c80e0D76d32939F,90720683,90.720683,1775813999,2026-04-10 09:39:59+00:00,2026-04-10
4,3ce25f8ca5a3356772c29b5550a16ce8208e79f489239f...,24848486,60,0x000000000004444c5dc75cB358380D2e3dE08A90,0xC165C9A4Db89aa72b23f66b96731A960084a1FE2,86845060,86.845060,1775813999,2026-04-10 09:39:59+00:00,2026-04-10


In [3]:
daily_volume = compute_daily_volume(df)
daily_tx = compute_daily_tx_count(df)

print("daily_volume:")
display(daily_volume.head())

print("daily_tx_count:")
display(daily_tx.head())

daily_volume:


,date,daily_volume_usdc
0,2026-04-10,4.264486e+08


daily_tx_count:


,date,daily_tx_count
0,2026-04-10,10042


In [4]:
fig_vol = go.Figure(
    data=[
        go.Scatter(
            x=daily_volume["date"].astype(str),
            y=daily_volume["daily_volume_usdc"],
            mode="lines+markers",
            name="每日转账量",
        )
    ]
)
fig_vol.update_layout(title="USDC 每日转账总量（样本窗口）", xaxis_title="日期(UTC)", yaxis_title="USDC")
fig_vol.show()

fig_tx = go.Figure(
    data=[
        go.Bar(
            x=daily_tx["date"].astype(str),
            y=daily_tx["daily_tx_count"],
            name="每日日志条数",
        )
    ]
)
fig_tx.update_layout(title="USDC 每日 Transfer 日志条数", xaxis_title="日期(UTC)", yaxis_title="条数")
fig_tx.show()

## 小结

- `daily_volume` 回答：**某一天链上 USDC 转账“总量”**（按日志汇总）
- `daily_tx_count` 回答：**某一天发生了多少条 USDC Transfer 日志**

下一步你可以尝试：
- 换一个更长的时间窗口（重新跑 week2 导出更大 CSV）
- 指定地址做净流入与对手方分析（`week3_analysis.py --address`）

# Week 3：USDC 每日转账量与日志笔数（探索）

本 Notebook 与 `analyzer.py`、`week3/week3_analysis.py` 使用**同一套 CSV 列与分析函数**，便于对照脚本导出的 `data/week3/*.csv`。

**口径提示**：`daily_tx_count` 为 Transfer **日志条数**，不等于链上「交易笔数」。

In [5]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

# 项目根目录（notebooks/ 的上级）
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analyzer import (
    compute_daily_tx_count,
    compute_daily_volume,
    ensure_datetime,
    load_usdc_csv,
)

DATA_DIR = ROOT / "data"
candidates = sorted(DATA_DIR.glob("usdc_transfers_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not candidates:
    raise FileNotFoundError(f"未找到 {DATA_DIR / 'usdc_transfers_*.csv'}，请先运行 Week 2 导出。")
csv_path = candidates[0]
print("使用文件:", csv_path)

raw = load_usdc_csv(csv_path)
df = ensure_datetime(raw)
daily_volume = compute_daily_volume(df)
daily_tx = compute_daily_tx_count(df)

daily = daily_volume.merge(daily_tx, on="date", how="outer").sort_values("date")
daily.head()

使用文件: D:\code\claude_code\Blockchain_learning\data\usdc_transfers_1775815222.csv


,date,daily_volume_usdc,daily_tx_count
0,2026-04-10,4.264486e+08,10042


In [6]:
fig_vol = px.line(
    daily,
    x="date",
    y="daily_volume_usdc",
    title="每日 USDC 转账总量（日志金额求和，UTC 日历日）",
    markers=True,
)
fig_vol.update_layout(xaxis_title="日期 (UTC)", yaxis_title="USDC")
fig_vol.show()

In [7]:
fig_tx = px.bar(
    daily,
    x="date",
    y="daily_tx_count",
    title="每日 Transfer 日志条数",
)
fig_tx.update_layout(xaxis_title="日期 (UTC)", yaxis_title="条数")
fig_tx.show()

## 学习小结（请自行填写）

- 样本时间范围内，总量与峰值日是否与 `week3_analysis.py` 控制台摘要一致？
- 若只观察一天，折线与柱状图分别说明了什么？
- 尝试将 `--address` 传入脚本后，对照 Notebook 中对同一地址用 `analyze_address_activity` 的结果。